# CPI ARIMA Baseline — Lv1

**Goal**: Fit ARIMA on historical CPI YoY data and evaluate walk-forward MAE.

**Steps**
1. Load CPI from FRED
2. Stationarity test (ADF)
3. ACF/PACF plots → choose ARIMA(p,d,q)
4. Fit and forecast
5. Walk-forward MAE vs Bloomberg consensus

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from src.models.arima import load_cpi, ARIMAForecaster

In [ ]:
cpi = load_cpi(start='2000-01-01')
cpi.plot(title='CPI YoY (%)', figsize=(12, 4))
plt.axhline(2, color='red', linestyle='--', label='Fed 2% target')
plt.legend()
plt.show()

In [ ]:
# ADF test for stationarity
adf_result = adfuller(cpi.dropna())
print(f'ADF statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
print('Stationary' if adf_result[1] < 0.05 else 'Non-stationary → apply differencing')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(cpi.dropna(),  lags=24, ax=ax1, title='ACF')
plot_pacf(cpi.dropna(), lags=24, ax=ax2, title='PACF')
plt.tight_layout()
plt.show()

In [ ]:
model = ARIMAForecaster(order=(2, 1, 2))
model.fit(cpi)
fc = model.forecast(steps=3)
print('Forecast (next 3 months):')
print(fc.round(3))

In [ ]:
mae = model.mae(cpi, n_test=12)
print(f'Walk-forward MAE (12m): {mae:.4f}%p')